In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Master/LTAINC/lightweight-medical-model')
RESULTS_ROOT = PROJECT_DIR / 'model-profile-results'
PROFILE_CSV = RESULTS_ROOT / 'profile.csv'

assert (PROJECT_DIR / 'tests/test-performance/profile_models.py').is_file(), 'Thiếu performance profiler'
print('Project:', PROJECT_DIR)
print('Results:', RESULTS_ROOT)

In [ ]:
import os

%cd {PROJECT_DIR}
os.environ['PYTHONPATH'] = str(PROJECT_DIR)
print('PYTHONPATH:', os.environ['PYTHONPATH'])
# Bỏ comment nếu cần cập nhật code trước khi chạy.
# !git pull origin main

# Model performance profile

Notebook này chạy `tests/test-performance/profile_models.py` để đo số tham số, dung lượng state dict, latency và FLOPs cho các kiến trúc BUSI.

In [ ]:
!pip install -q pandas matplotlib

## Chạy profiler

Mặc định dùng CPU để số latency dễ so sánh giữa các lần chạy. Nếu muốn đo GPU latency, đổi `DEVICE = 'cuda'`.

In [ ]:
MODELS = ['mednet', 'mk_mnet', 'r_cbam_mnet']
WIDTH_MULTS = [0.25, 0.5, 1.0]
IMAGE_SIZE = 224
WARMUP = 10
ITERATIONS = 50
DEVICE = 'cpu'

model_args = ' '.join(MODELS)
width_args = ' '.join(map(str, WIDTH_MULTS))
print('Models:', model_args)
print('Width multipliers:', width_args)

In [ ]:
!python tests/test-performance/profile_models.py \
  --models $model_args \
  --width-mults $width_args \
  --image-size $IMAGE_SIZE \
  --warmup $WARMUP \
  --iterations $ITERATIONS \
  --device $DEVICE \
  --output "$PROFILE_CSV"

## Bảng kết quả

In [ ]:
import pandas as pd
from IPython.display import display

profile = pd.read_csv(PROFILE_CSV)
display(profile)

display(profile[[
    'model', 'width_mult', 'params_m', 'trainable_params_m',
    'state_size_mb', 'cpu_or_device_latency_ms', 'profiler_gflops'
]].sort_values(['model', 'width_mult'], na_position='last'))

## Vẽ nhanh latency / cost

In [ ]:
import matplotlib.pyplot as plt

plot_df = profile.copy()
plot_df['label'] = plot_df.apply(
    lambda row: f"{row['model']} w={row['width_mult']}" if pd.notna(row['width_mult']) else row['model'],
    axis=1,
)

ax = plot_df.plot.scatter(
    x='params_m',
    y='cpu_or_device_latency_ms',
    figsize=(7, 4.5),
    grid=True,
)
for _, row in plot_df.iterrows():
    ax.annotate(row['label'], (row['params_m'], row['cpu_or_device_latency_ms']), fontsize=8)
ax.set_xlabel('Parameters (M)')
ax.set_ylabel(f'Latency ({DEVICE}, ms)')
ax.set_title('Model size vs latency')
plt.tight_layout()
plot_path = RESULTS_ROOT / 'params_latency.png'
plt.savefig(plot_path, dpi=180)
plt.show()
print('Saved:', plot_path)